In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge — DL & GenAI Project

**Roll No:** 23f2004827

## Objective
Predict the top-3 most likely correct answers (ranked) for multiple-choice
questions, evaluated using Mean Average Precision at 3 (MAP@3).

## Approach
Four models are built and compared:
1. TF-IDF + cosine similarity (baseline, no training)
2. Pretrained sentence-transformer embeddings + cosine similarity
3. Fine-tuned BERT (pretrained, fine-tuned for multiple-choice)
4. Feedforward neural network trained from scratch on TF-IDF features

Final predictions use a weighted ensemble of models 2–4, with weights
selected via grid search on a held-out validation set.

## 1. Data Loading
Loading train and test sets from the competition data.

In [7]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head(2)

Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


## 2. Evaluation Metric
Implementing MAP@3 locally so we can evaluate models on our own
validation split before submitting to Kaggle. A quick sanity check
confirms the function behaves as expected on toy inputs.

In [8]:
def map_at_3(predictions, true_answers):
    
    scores = []
    for pred, true in zip(predictions, true_answers):
        pred_list = pred.split()
        if true in pred_list:
            rank = pred_list.index(true) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return np.mean(scores)


test_preds = ["A B C", "B A C", "C D A"]
test_true  = ["A", "A", "A"]
print("Sanity check (expect ~0.5):", map_at_3(test_preds, test_true))

Sanity check (expect ~0.5): 0.611111111111111


## 3. Model 1 — TF-IDF + Cosine Similarity (Baseline)
This baseline represents questions and answer options using TF-IDF vectors and ranks the options using cosine similarity.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_predict(df):
    
    option_cols = ['A', 'B', 'C', 'D', 'E']
    predictions = []

    
    all_text = df['prompt'].tolist()
    for col in option_cols:
        all_text += df[col].tolist()

    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    vectorizer.fit(all_text)

    prompt_vecs = vectorizer.transform(df['prompt'])

    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_vecs = vectorizer.transform(option_texts)
        sims = cosine_similarity(prompt_vecs[i], option_vecs)[0]  

        ranked_idx = np.argsort(sims)[::-1]  
        top3_letters = [option_cols[idx] for idx in ranked_idx[:3]]
        predictions.append(" ".join(top3_letters))

    return predictions


train_preds_model1 = tfidf_predict(train)
score_model1 = map_at_3(train_preds_model1, train['answer'].tolist())
print("Model 1 (TF-IDF) MAP@3 on train:", score_model1)

Model 1 (TF-IDF) MAP@3 on train: 0.27616666666666667


## 4. Experiment Tracking Setup
Logging in to Weights & Biases to track and compare MAP@3, accuracy,
and F1 across all models. The API key is loaded securely via Kaggle
Secrets rather than hardcoded.

In [10]:
!pip install wandb -q
import wandb
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2004827 (23f2004827-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
run = wandb.init(project="23f2004827-t22026", name="model1-tfidf-baseline")
wandb.log({"map@3": score_model1, "model": "tfidf_cosine_similarity"})
run.finish()

map@3,▁
map@3,0.27617
model,tfidf_cosine_similar...


## 5. Model 2 — Pretrained Sentence-Transformer
Using `all-MiniLM-L6-v2`, a pretrained sentence embedding model, to
generate context-aware embeddings for the prompt and each option.
Options are ranked by cosine similarity in embedding space — a stronger
representation than TF-IDF since it captures semantic meaning, not just
word overlap.

In [12]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer


model_st = SentenceTransformer('all-MiniLM-L6-v2')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
import torch

def sentence_transformer_predict(df, model_st, batch_size=32):
   
    option_cols = ['A', 'B', 'C', 'D', 'E']
    predictions = []

    
    prompt_embeddings = model_st.encode(df['prompt'].tolist(), batch_size=batch_size, show_progress_bar=True, convert_to_tensor=True)

    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_embeddings = model_st.encode(option_texts, convert_to_tensor=True)

        
        sims = torch.nn.functional.cosine_similarity(
            prompt_embeddings[i].unsqueeze(0), option_embeddings
        )

        ranked_idx = torch.argsort(sims, descending=True).tolist()
        top3_letters = [option_cols[idx] for idx in ranked_idx[:3]]
        predictions.append(" ".join(top3_letters))

    return predictions

train_preds_model2 = sentence_transformer_predict(train, model_st)
score_model2 = map_at_3(train_preds_model2, train['answer'].tolist())
print("Model 2 (Sentence-Transformer) MAP@3 on train:", score_model2)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Model 2 (Sentence-Transformer) MAP@3 on train: 0.4230833333333333


In [14]:
run = wandb.init(project="23f2004827-t22026", name="model2-sentence-transformer")
wandb.log({"map@3": score_model2, "model": "all-MiniLM-L6-v2_cosine_similarity"})
run.finish()

map@3,▁
map@3,0.42308
model,all-MiniLM-L6-v2_cos...


## 6. Model 3 — Fine-tuned BERT (Pretrained Model)
Formatting the MCQ task for `bert-base-uncased` using the multiple-choice
head, tokenizing (prompt, option) pairs for each of the 5 options, and
splitting into train/validation sets for fine-tuning.

In [15]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
import torch

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
option_cols = ['A', 'B', 'C', 'D', 'E']

def preprocess(df):
    first_sentences = [[p]*5 for p in df['prompt']]
    second_sentences = [[df.iloc[i][col] for col in option_cols] for i in range(len(df))]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(first_sentences, second_sentences, truncation=True, padding='max_length', max_length=256)
    features = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
    return features

train_split, val_split = train_test_split(train, test_size=0.15, random_state=42)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

train_features = preprocess(train_split)
val_features = preprocess(val_split)

print("Train samples:", len(train_split), "Val samples:", len(val_split))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train samples: 1700 Val samples: 300


In [16]:
class MCQDataset(torch.utils.data.Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.features.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

label_map = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
train_labels = [label_map[a] for a in train_split['answer']]
val_labels = [label_map[a] for a in val_split['answer']]

train_dataset = MCQDataset(train_features, train_labels)
val_dataset = MCQDataset(val_features, val_labels)

model3 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

print("Dataset ready. Train size:", len(train_dataset), "Val size:", len(val_dataset))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset ready. Train size: 1700 Val size: 300
